In [1]:
from pathlib import Path
import pandas as pd

sample_path = Path("../data/processed/sample_jobs.csv")
jobs = pd.read_csv(sample_path, dtype={"job_id": str})

print("Number of jobs:", len(jobs))
print("Missing descriptions:", jobs["description"].isna().sum())

jobs[["job_id", "title", "location"]].head()

Number of jobs: 500
Missing descriptions: 0


,job_id,title,location
0,3901384985,Underwriter II / Senior Underwriter,"New York, United States"
1,3904954171,Lead Customer service Representative,"Hickory, NC"
2,3844654703,Account Manager,"Los Angeles, CA"
3,3904991666,Nurse Assistant - 6W Cardiac Stepdown - Full T...,"Mason City, IA"
4,3905324896,Psychology Technician - Department of Health P...,"Columbia, MO"


In [2]:
candidate = {
    "candidate_id": "synthetic_data_analyst",
    "skills": ["Python", "SQL", "Excel", "data analysis", "statistics"],
    "education": "Bachelor's degree in mathematics",
    "experience": (
        "Entry-level candidate with academic projects in data analysis. "
        "Used Python and SQL to clean data, analyze datasets, "
        "and communicate findings through reports and visualizations."
    ),
}

candidate_text = " ".join([
    " ".join(candidate["skills"]),
    candidate["education"],
    candidate["experience"],
])

print(candidate_text)

Python SQL Excel data analysis statistics Bachelor's degree in mathematics Entry-level candidate with academic projects in data analysis. Used Python and SQL to clean data, analyze datasets, and communicate findings through reports and visualizations.


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Fit the vocabulary and word weights on job descriptions only.
vectorizer = TfidfVectorizer(stop_words="english")
job_vectors = vectorizer.fit_transform(jobs["description"])

# Represent the candidate using the same vocabulary and weights.
candidate_vector = vectorizer.transform([candidate_text])

# Compare this candidate with every job.
text_scores = cosine_similarity(candidate_vector, job_vectors).ravel()

results = jobs[["job_id", "title", "location"]].copy()
results["text_score"] = text_scores

top_10 = results.sort_values(
    ["text_score", "job_id"],
    ascending=[False, True],
).head(10)

top_10

,job_id,title,location,text_score
161,3902343369,Data Engineer,United States,0.271928
65,3894544709,Senior Data Analyst,"McLean, VA",0.243944
138,3903457662,Data Entry clerk.,United States,0.228296
262,3905302758,IT Data Engineer,"Atlanta, GA",0.187395
116,3901642507,Statistician,"Charlotte, NC",0.176310
476,3905215190,Azure Data Architect,"Roseville, CA",0.162160
206,3901941388,Sr. Data Engineer (onsite),"Bloomington, IN",0.161382
365,3894546475,Data Manager and Analytical Support Specialist...,"Bronx, NY",0.160081
6,3905256395,Sales Operations Specialist,"Columbia, MD",0.159647
250,3904958394,Technical Lead Information Intelligent Cloud S...,"Orlando, FL",0.158404


In [4]:
for job_id in top_10["job_id"].head(3):
    job = jobs.loc[jobs["job_id"] == job_id].iloc[0]

    print(job["title"])
    print(str(job["description"])[:2500])
    print("\n" + "=" * 80 + "\n")

Data Engineer
Work with teams and stakeholders across the business to understand data needs to design performant, scalable, and low-technical-debt solutions in a multi-cloud environment.Specialize in Azure data offerings like SQL MI, Redis Cache, and PostgreSQL.Utilize cloud-native services for data storage, processing, and analytics.Deploy data integration processes that are efficient and reliable and maximize data flow across the organization.Implement security protocols to safeguard sensitive data and enforce data governance.Boost data architectures for performance and scalability.Validate data quality and consistency throughout the data lifecycle.Build strong partnerships with product managers, developers, and other stakeholders to understand requirements and provide technical guidance.Key Qualifications:Bachelor's degree in computer science or similar degree preferred.3+ years of hands-on experience with cloud providers required.Proficient in SQL, PostGres, ElasticSearch, Redis an

In [5]:
candidates = [
    candidate,
    {
        "candidate_id": "synthetic_accountant",
        "skills": [
            "accounting", "Excel", "financial reporting",
            "bookkeeping", "account reconciliation",
        ],
        "education": "Bachelor's degree in accounting",
        "experience": (
            "Two years of accounting experience preparing financial "
            "reports, reconciling accounts, and processing invoices."
        ),
    },
    {
        "candidate_id": "synthetic_marketing_specialist",
        "skills": [
            "digital marketing", "SEO", "Google Analytics",
            "content creation", "social media",
        ],
        "education": "Bachelor's degree in marketing",
        "experience": (
            "Two years of marketing experience managing social media "
            "campaigns, writing content, and analyzing campaign performance."
        ),
    },
]

print([profile["candidate_id"] for profile in candidates])

['synthetic_data_analyst', 'synthetic_accountant', 'synthetic_marketing_specialist']


In [6]:
def profile_to_text(profile):
    """Combine declared candidate information into one text."""
    return " ".join([
        " ".join(profile["skills"]),
        profile["education"],
        profile["experience"],
    ])


rankings = {}

for profile in candidates:
    profile_vector = vectorizer.transform([profile_to_text(profile)])
    scores = cosine_similarity(profile_vector, job_vectors).ravel()

    ranked_jobs = jobs[["job_id", "title", "location"]].copy()
    ranked_jobs["text_score"] = scores

    rankings[profile["candidate_id"]] = ranked_jobs.sort_values(
        ["text_score", "job_id"],
        ascending=[False, True],
    ).head(10)


for candidate_id, ranking in rankings.items():
    print(candidate_id)
    display(ranking)

synthetic_data_analyst


,job_id,title,location,text_score
161,3902343369,Data Engineer,United States,0.271928
65,3894544709,Senior Data Analyst,"McLean, VA",0.243944
138,3903457662,Data Entry clerk.,United States,0.228296
262,3905302758,IT Data Engineer,"Atlanta, GA",0.187395
116,3901642507,Statistician,"Charlotte, NC",0.176310
476,3905215190,Azure Data Architect,"Roseville, CA",0.162160
206,3901941388,Sr. Data Engineer (onsite),"Bloomington, IN",0.161382
365,3894546475,Data Manager and Analytical Support Specialist...,"Bronx, NY",0.160081
6,3905256395,Sales Operations Specialist,"Columbia, MD",0.159647
250,3904958394,Technical Lead Information Intelligent Cloud S...,"Orlando, FL",0.158404


synthetic_accountant


,job_id,title,location,text_score
79,3904363876,Accounting Manager,"Oklahoma City, OK",0.501731
74,3901392467,Finance Coordinator,"Boston, MA",0.340185
370,3905883406,Senior Accountant,Topeka Metropolitan Area,0.317252
462,3902758248,Accounting Manager,"Salt Lake City, UT",0.274265
141,3905372329,"Associate Manager, Hedge Fund/Private Equity F...","Edina, MN",0.218016
28,3898161791,Staff Accountant,"Cedar Rapids, IA",0.203656
473,3901391232,Accounts Payable Specialist,"Detroit, MI",0.195322
307,3906256243,VP Corporate Financial Planning Analysis,"Pittsburgh, PA",0.183776
20,3900079355,Accounts Payable Analyst,"Kansas City, MO",0.174219
13,3900063970,Vice President Finance,"Pittsburgh, PA",0.173512


synthetic_marketing_specialist


,job_id,title,location,text_score
394,3894893823,Content Specialist,"Austin, TX",0.375379
214,3905669377,Public Affairs Specialist,"Washington, DC",0.322000
263,3904091394,Hiring- Web & Mobile Content Specialist- Merr...,"Merrifield, VA",0.281199
5,3895232336,Program Manager,"Charleston, South Carolina Metropolitan Area",0.268624
420,3891291458,Design Internship,"Chicago, IL",0.207588
352,3904430618,Marketing Specialist,"Fort Lauderdale, FL",0.203658
248,3905354157,Digital Marketing Client Account Manager,"Arizona, United States",0.193400
26,3905886911,Marketing Assistant,"Savannah, GA",0.192164
281,3894980127,Senior Account Executive – Public Relations (F...,"New York, NY",0.186911
198,3901652587,Vice President of Sales & Marketing - Southern...,"Richmond, VA",0.170515


In [7]:
import sys
from pathlib import Path
import importlib
import numpy as np

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import src.matching as matching
importlib.reload(matching)

matcher = matching.TextMatcher().fit(jobs)
module_scores = matcher.score(candidates[0])

np.testing.assert_allclose(module_scores, text_scores)
print("OK: module scores match the original notebook scores.")

OK: module scores match the original notebook scores.


In [8]:
import src.recommendation as recommendation
importlib.reload(recommendation)

for profile in candidates:
    print(profile["candidate_id"])

    top_jobs = recommendation.recommend_top_k(
        profile, jobs, k=10, matcher=matcher
    )

    display(top_jobs[["job_id", "title", "location", "text_score"]])

synthetic_data_analyst


,job_id,title,location,text_score
0,3902343369,Data Engineer,United States,0.271928
1,3894544709,Senior Data Analyst,"McLean, VA",0.243944
2,3903457662,Data Entry clerk.,United States,0.228296
3,3905302758,IT Data Engineer,"Atlanta, GA",0.187395
4,3901642507,Statistician,"Charlotte, NC",0.176310
5,3905215190,Azure Data Architect,"Roseville, CA",0.162160
6,3901941388,Sr. Data Engineer (onsite),"Bloomington, IN",0.161382
7,3894546475,Data Manager and Analytical Support Specialist...,"Bronx, NY",0.160081
8,3905256395,Sales Operations Specialist,"Columbia, MD",0.159647
9,3904958394,Technical Lead Information Intelligent Cloud S...,"Orlando, FL",0.158404


synthetic_accountant


,job_id,title,location,text_score
0,3904363876,Accounting Manager,"Oklahoma City, OK",0.501731
1,3901392467,Finance Coordinator,"Boston, MA",0.340185
2,3905883406,Senior Accountant,Topeka Metropolitan Area,0.317252
3,3902758248,Accounting Manager,"Salt Lake City, UT",0.274265
4,3905372329,"Associate Manager, Hedge Fund/Private Equity F...","Edina, MN",0.218016
5,3898161791,Staff Accountant,"Cedar Rapids, IA",0.203656
6,3901391232,Accounts Payable Specialist,"Detroit, MI",0.195322
7,3906256243,VP Corporate Financial Planning Analysis,"Pittsburgh, PA",0.183776
8,3900079355,Accounts Payable Analyst,"Kansas City, MO",0.174219
9,3900063970,Vice President Finance,"Pittsburgh, PA",0.173512


synthetic_marketing_specialist


,job_id,title,location,text_score
0,3894893823,Content Specialist,"Austin, TX",0.375379
1,3905669377,Public Affairs Specialist,"Washington, DC",0.322000
2,3904091394,Hiring- Web & Mobile Content Specialist- Merr...,"Merrifield, VA",0.281199
3,3895232336,Program Manager,"Charleston, South Carolina Metropolitan Area",0.268624
4,3891291458,Design Internship,"Chicago, IL",0.207588
5,3904430618,Marketing Specialist,"Fort Lauderdale, FL",0.203658
6,3905354157,Digital Marketing Client Account Manager,"Arizona, United States",0.193400
7,3905886911,Marketing Assistant,"Savannah, GA",0.192164
8,3894980127,Senior Account Executive – Public Relations (F...,"New York, NY",0.186911
9,3901652587,Vice President of Sales & Marketing - Southern...,"Richmond, VA",0.170515


In [9]:
importlib.reload(matching)

matching.skill_coverage(
    candidate_skills=["Python", "SQL", "Excel"],
    job_skills=["Python", "SQL", "AWS"],
)

{'skill_score': 0.6666666666666666,
 'matched_skills': ['python', 'sql'],
 'unverified_skills': ['aws']}

In [10]:
import re

# Temporary demo vocabulary, not a validated skill extractor.
DEMO_SKILL_PATTERNS = {
    "python": r"\bpython\b",
    "sql": r"\bsql\b",
    "excel": r"\b(?:microsoft\s+)?excel\b",
    "data analysis": r"\bdata analysis\b",
    "statistics": r"\bstatistics\b",
    "aws": r"\baws\b|\bamazon web services\b",
    "azure": r"\bazure\b",
    "accounting": r"\baccounting\b",
    "financial reporting": r"\bfinancial reporting\b",
    "bookkeeping": r"\bbookkeeping\b",
    "account reconciliation": r"\baccount reconciliations?\b",
    "digital marketing": r"\bdigital marketing\b",
    "seo": r"\bseo\b|\bsearch engine optimization\b",
    "google analytics": r"\bgoogle analytics\b",
    "content creation": r"\bcontent creation\b",
    "social media": r"\bsocial media\b",
}


def extract_demo_skills(description):
    """Find vocabulary mentions without interpreting requirements."""
    return [
        skill
        for skill, pattern in DEMO_SKILL_PATTERNS.items()
        if re.search(pattern, description, flags=re.IGNORECASE)
    ]


jobs["demo_skills"] = jobs["description"].apply(extract_demo_skills)

print(
    "Jobs with at least one vocabulary match:",
    jobs["demo_skills"].map(bool).sum(),
    "out of",
    len(jobs),
)

jobs[["title", "demo_skills"]].head(10)

Jobs with at least one vocabulary match: 150 out of 500


,title,demo_skills
0,Underwriter II / Senior Underwriter,[excel]
1,Lead Customer service Representative,[]
2,Account Manager,[]
3,Nurse Assistant - 6W Cardiac Stepdown - Full T...,[]
4,Psychology Technician - Department of Health P...,[]
5,Program Manager,"[excel, digital marketing, google analytics]"
6,Sales Operations Specialist,[excel]
7,"Director, Facilities",[]
8,Cosmic Bingo Host,[]
9,Associate,[]


In [11]:
# Use reusable scoring functions with temporary demo skill labels.
importlib.reload(matching)
importlib.reload(recommendation)
matcher = matching.TextMatcher().fit(jobs)
demo_rankings = {}

for profile in candidates:
    demo_rankings[profile["candidate_id"]] = recommendation.score_jobs(
        profile, jobs, matcher=matcher,
        skills_column="demo_skills", text_weight=0.5,
    )
    print(profile["candidate_id"])
    for method in ["text", "skill", "hybrid"]:
        print(method)
        top_jobs = recommendation.recommend_top_k(
            profile, jobs, k=10, matcher=matcher, method=method,
            skills_column="demo_skills", text_weight=0.5,
        )
        display(top_jobs[[
            "job_id", "title", "text_score", "skill_score", "hybrid_score",
            "matched_skills", "unverified_skills", "hybrid_uses_skills",
        ]])

synthetic_data_analyst
text


,job_id,title,text_score,skill_score,hybrid_score,matched_skills,unverified_skills,hybrid_uses_skills
0,3902343369,Data Engineer,0.271928,0.500000,0.385964,"[python, sql]","[aws, azure]",True
1,3894544709,Senior Data Analyst,0.243944,0.666667,0.455305,"[python, sql]",[aws],True
2,3903457662,Data Entry clerk.,0.228296,NaN,0.228296,[],[],False
3,3905302758,IT Data Engineer,0.187395,1.000000,0.593697,[python],[],True
4,3901642507,Statistician,0.176310,1.000000,0.588155,"[data analysis, excel, python, sql, statistics]",[],True
5,3905215190,Azure Data Architect,0.162160,0.500000,0.331080,[sql],[azure],True
6,3901941388,Sr. Data Engineer (onsite),0.161382,0.500000,0.330691,[python],[azure],True
7,3894546475,Data Manager and Analytical Support Specialist...,0.160081,1.000000,0.580041,"[excel, sql]",[],True
8,3905256395,Sales Operations Specialist,0.159647,1.000000,0.579823,[excel],[],True
9,3904958394,Technical Lead Information Intelligent Cloud S...,0.158404,1.000000,0.579202,[sql],[],True


skill


,job_id,title,text_score,skill_score,hybrid_score,matched_skills,unverified_skills,hybrid_uses_skills
0,3884434211,Quality Assurance Engineer (323816),0.049750,1.0,0.524875,[sql],[],True
1,3884443702,Correspondence Management Professional,0.010150,1.0,0.505075,[excel],[],True
2,3884446482,Product Data Hub Developer,0.111768,1.0,0.555884,[sql],[],True
3,3884927872,Facilities Technician II,0.029127,1.0,0.514564,[statistics],[],True
4,3884937059,Supply Chain/Procurement Manager,0.019229,1.0,0.509615,[excel],[],True
5,3885106278,Quality Control Technician - 1st Shift (Auburn...,0.037078,1.0,0.518539,[excel],[],True
6,3885858721,Operations Analyst - US Government,0.029335,1.0,0.514667,[excel],[],True
7,3887487791,Sales Administrative Assistant,0.032014,1.0,0.516007,[excel],[],True
8,3887702108,"Systems Analysis, Senior",0.087285,1.0,0.543643,[excel],[],True
9,3887878425,On-Air Scheduling Specialist,0.007147,1.0,0.503574,[excel],[],True


hybrid


,job_id,title,text_score,skill_score,hybrid_score,matched_skills,unverified_skills,hybrid_uses_skills
0,3905302758,IT Data Engineer,0.187395,1.0,0.593697,[python],[],True
1,3901642507,Statistician,0.176310,1.0,0.588155,"[data analysis, excel, python, sql, statistics]",[],True
2,3894546475,Data Manager and Analytical Support Specialist...,0.160081,1.0,0.580041,"[excel, sql]",[],True
3,3905256395,Sales Operations Specialist,0.159647,1.0,0.579823,[excel],[],True
4,3904958394,Technical Lead Information Intelligent Cloud S...,0.158404,1.0,0.579202,[sql],[],True
5,3888475408,Senior Data and Reporting Professional,0.153960,1.0,0.576980,"[data analysis, excel, sql, statistics]",[],True
6,3901937696,Frontend Developer,0.141743,1.0,0.570872,"[python, sql]",[],True
7,3903474861,Business Strategy Analyst Senior (P&C Member ...,0.125117,1.0,0.562558,"[data analysis, excel, python, sql]",[],True
8,3904946918,OnBase Developer,0.113339,1.0,0.556669,[sql],[],True
9,3884446482,Product Data Hub Developer,0.111768,1.0,0.555884,[sql],[],True


... (8 outputs left out)

skill


,job_id,title,text_score,skill_score,hybrid_score,matched_skills,unverified_skills,hybrid_uses_skills
0,3889114336,Wordpress Developer,0.021699,1.0,0.510849,[seo],[],True
1,3891291458,Design Internship,0.207588,1.0,0.603794,[social media],[],True
2,3894864954,Assistant Property Manager,0.032021,1.0,0.516010,[social media],[],True
3,3894893823,Content Specialist,0.375379,1.0,0.687689,"[seo, social media]",[],True
4,3894980127,Senior Account Executive – Public Relations (F...,0.186911,1.0,0.593455,[social media],[],True
5,3895590829,Network Development Specialist,0.014085,1.0,0.507043,[social media],[],True
6,3899546043,Entry Level Communications Representative,0.084253,1.0,0.542127,[social media],[],True
7,3901652587,Vice President of Sales & Marketing - Southern...,0.170515,1.0,0.585257,[social media],[],True
8,3902365634,SEO Writer (Article writing internship),0.098751,1.0,0.549375,[seo],[],True
9,3902941559,Graphic Designer,0.101076,1.0,0.550538,[social media],[],True


hybrid


,job_id,title,text_score,skill_score,hybrid_score,matched_skills,unverified_skills,hybrid_uses_skills
0,3894893823,Content Specialist,0.375379,1.0,0.687689,"[seo, social media]",[],True
1,3905669377,Public Affairs Specialist,0.322000,1.0,0.661000,"[content creation, social media]",[],True
2,3904091394,Hiring- Web & Mobile Content Specialist- Merr...,0.281199,1.0,0.640600,"[google analytics, seo]",[],True
3,3891291458,Design Internship,0.207588,1.0,0.603794,[social media],[],True
4,3905354157,Digital Marketing Client Account Manager,0.193400,1.0,0.596700,"[digital marketing, social media]",[],True
5,3894980127,Senior Account Executive – Public Relations (F...,0.186911,1.0,0.593455,[social media],[],True
6,3901652587,Vice President of Sales & Marketing - Southern...,0.170515,1.0,0.585257,[social media],[],True
7,3906256452,"Executive Communications Director, Marketing",0.155903,1.0,0.577951,[social media],[],True
8,3902941559,Graphic Designer,0.101076,1.0,0.550538,[social media],[],True
9,3902365634,SEO Writer (Article writing internship),0.098751,1.0,0.549375,[seo],[],True


## Baseline limitations

- Recommendations use a reproducible sample of 500 historical postings.
- Candidate profiles are synthetic.
- TF-IDF measures text similarity; experience and mandatory qualifications
  are not checked.
- Skill matching uses a temporary, limited vocabulary for demonstration.
  It detects mentions, not necessarily requirements.
- A skill score of 1 means coverage of detected vocabulary terms only.
- Hybrid scoring uses unvalidated equal weights. When no skills are
  detected, it falls back to text similarity.
- No human relevance evaluation has been performed. These results do not
  establish that hybrid ranking is better than either baseline.
- Scores are not probabilities of interviews or offers.